# Getting Started

This example trains a small decision tree, ports it to JavaScript with
[sklearn-porter](https://github.com/nok/sklearn-porter), runs it with
Node.js and checks that it predicts the same classes as scikit-learn.

Running the ported classifier requires `node`.

## Train a classifier

In [1]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)
clf = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)

## Port the classifier

`port` returns the source code. With the template `combined` the splits of
the tree become nested if-else statements:

In [2]:
from sklearn_porter import port

print(port(clf, language='js', template='combined'))

/*
This file is generated by https://github.com/nok/sklearn-porter/

Estimator:
	DecisionTreeClassifier
	https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

Environment:
	scikit-learn v1.9.1
	sklearn-porter v1.0.0

Usage:
	1. Execute a prediction:
		$ node DecisionTreeClassifier.js feature_1 ... feature_4
	2. Execute the predictions of several samples (one per line):
		$ node DecisionTreeClassifier.js < samples.txt
	3. Use the class in Node or in a browser:
		var DecisionTreeClassifier = require('./DecisionTreeClassifier.js');  // Node
		// or <script src="DecisionTreeClassifier.js"></script> in a browser
		var estimator = new DecisionTreeClassifier();
		estimator.predict([feature_1, ..., feature_4]);
*/
(function () {
'use strict';

var DecisionTreeClassifier = function() {

	var _findMax = function(nums) {
		var i = 0, l = nums.length, idx = 0;
		for (; i < l; i++) {
			idx = nums[i] > nums[idx] ? i : idx;
		}
		return idx;
	};

	// Copy the cl

## Run the ported classifier

Save the source code to a file and pass the features of a sample:

In [3]:
from sklearn_porter import save

save(clf, language='js', template='combined', directory='/tmp')

'/tmp/DecisionTreeClassifier.js'

In [4]:
!node /tmp/DecisionTreeClassifier.js 6.3 3.3 6.0 2.5

Predicted class: 2
Probability of class #0 : 0
Probability of class #1 : 0.021739130434782608
Probability of class #2 : 0.9782608695652174


Or use the class in your own code, e.g. with `require` in Node.js or with
a `<script>` tag in a browser:

In [5]:
!node -e "const Tree = require('/tmp/DecisionTreeClassifier.js'); console.log('Class: ' + new Tree().predict([6.3, 3.3, 6.0, 2.5]))"

Class: 2


## Make predictions from Python

`make` runs the ported classifier with several samples and returns the
predicted classes and the probabilities:

In [6]:
from sklearn_porter import make

y_pred, y_proba = make(clf, X[[0, 50, 100]], language='js')

print(y_pred)
print(y_proba)

[0 1 2]
[[1.         0.         0.        ]
 [0.         0.90740741 0.09259259]
 [0.         0.02173913 0.97826087]]


## Check the ported classifier

`test` compares the predictions of the ported classifier with those of
scikit-learn. The score is the fraction of equal predictions, so `1.0`
means that all predictions match. Always check the ported classifier with
representative samples:

In [7]:
from sklearn_porter import test

print(test(clf, X, language='js'))

1.0


## Templates and the class `Porter`

The template defines how the model data is stored: `attached` (as arrays in
the code), `combined` (in the logic of the code) or `exported` (in a
separate JSON file, so you can update the model without changing the
code). The class `Porter` keeps the language and the template for several
calls:

In [8]:
from pathlib import Path

from sklearn_porter import Porter

porter = Porter(clf, language='js', template='exported')
src_path, json_path = porter.save(directory='/tmp')

print(Path(json_path).read_text())
print(porter.test(X))

{"lefts":[-1,-2],"rights":[1,-3],"thresholds":[0.8000000417232512,1.7500000596046448],"indices":[3,3],"missing_go_to_left":[0,1],"classes":[1.0,0.0,0.0,0.0,0.9074074074074074,0.09259259259259259,0.0,0.021739130434782608,0.9782608695652174]}
1.0


## Next steps

- [Transformers](transformers.ipynb): port the preprocessing, e.g. a
  `StandardScaler`.
- [Pipelines](pipelines.ipynb): port the preprocessing and the classifier
  of a pipeline to one source file.
- [CLI](cli.ipynb): port a saved classifier with the command `porter`.